In [ ]:
#Importing Libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
sns.set()
%matplotlib inline

from sklearn.model_selection import train_test_split

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"


In [ ]:
# Loading The Dataset
df = pd.read_csv('Reviews.csv')

In [ ]:
# Viewing The first 5 rows
df.head()

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
df.info()

In [ ]:
final_dataset = df[df.HelpfulnessDenominator <= df.HelpfulnessDenominator]

In [ ]:
# Checking Missing Values
df.isnull().sum()

In [ ]:
# Handling The Missing Values
df['ProfileName'] = df['ProfileName'].fillna(df['ProfileName'].mode()[0])
df['Summary'] = df['Summary'].fillna(df['Summary'].mode()[0])

In [ ]:
# Text Preprocessing
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

In [ ]:
from tqdm import tqdm
import re
from bs4 import BeautifulSoup
from nltk.corpus import stopwords
preprocessed_review = []

for sentence in tqdm(final_dataset['Text'].values):
    sentence = re.sub(r"http\S+", "", sentence)
    sentence = BeautifulSoup(sentence, 'lxml').get_text()
    sentence = re.sub('[^a-zA-Z]+', ' ', sentence)
    sentence = ' '.join(
        e.lower() for e in sentence.split()
        if e.lower() not in stop_words
    )
    preprocessed_review.append(sentence.strip())


In [ ]:
final_dataset['clean_text'] = preprocessed_review


In [ ]:
final_dataset = final_dataset.drop('Text', axis=1)

In [ ]:
final_dataset.head()

In [ ]:
!pip install transformers datasets tensorflow -q
!pip install evaluate


In [ ]:
final_dataset = final_dataset.sample(n=250000, random_state=42)

In [ ]:
final_dataset['Score'] = final_dataset['Score'] - 1

In [ ]:
train_df, test_df = train_test_split(
    final_dataset, test_size=0.2, stratify=final_dataset['Score'], random_state=42
)

val_df, test_df = train_test_split(
    test_df, test_size=0.5, stratify=test_df['Score'], random_state=42
)

In [ ]:
from transformers import DistilBertTokenizerFast
from datasets import Dataset

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch['clean_text'],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df).map(tokenize, batched=True)

train_ds = train_ds.rename_column("Score", "labels")
val_ds   = val_ds.rename_column("Score", "labels")
test_ds  = test_ds.rename_column("Score", "labels")

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

In [ ]:
from transformers import DistilBertForSequenceClassification
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=5
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./biner",

    # core settings
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    # optimizer / schedule helpers
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,

    # logging / saving
    logging_dir="./logs",
    logging_steps=200,
    save_steps=2000,        # save checkpoints periodically so you can resume
    save_total_limit=2,     # keep last 2 checkpoints
)

In [ ]:
import evaluate
import numpy as np

acc = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": acc.compute(predictions=preds, references=labels),
        "f1": f1.compute(predictions=preds, references=labels, average="macro")
    }

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate(test_ds)

In [ ]:

def predict_review(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to('cuda')
    outputs = model(**inputs)
    pred = torch.argmax(outputs.logits).item()
    return pred + 1   # convert back to 1–5 stars

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# True & Pred values (you already have these)
y_true = test_df['Score'].values
y_pred = np.argmax(preds.predictions, axis=1)

# Accuracy
accuracy = accuracy_score(y_true, y_pred)
print("Accuracy:", accuracy)

# Classification Report
report = classification_report(y_true, y_pred)
print("\nClassification Report:\n")
print(report)